# Le théorème central limite, mesuré

**Probas** (racine) — à côté de [`Probas-KLS-Concentration`](Probas-KLS-Concentration.ipynb).

Le **théorème central limite (TCL)** est l'un des résultats les plus puissants de toute la probabilité : la somme d'un grand nombre de variables aléatoires indépendantes, convenablement normalisée, devient **gaussienne** — quelle que soit la loi de départ, pourvu qu'elle ait une **variance finie**.

On ne se contente pas d'énoncer le théorème : on le **mesure**.

1. On mesure la **vitesse** de convergence (l'écart à la gaussienne décroît comme `1/√n`), et on vérifie que ce **taux est universel** — indépendant de la loi de départ.
2. On attaque l'hypothèse la moins visible : **la variance finie**. On montre par un contre-exemple à queue lourde (variance infinie) que le théorème **s'effondre** : la somme normalisée ne se stabilise plus en loi gaussienne. L'hypothèse n'est pas une formalité, elle est **porteuse**.

In [1]:
import numpy as np
import math

rng = np.random.default_rng(2026)

def phi(x):
    """Fonction de répartition de la loi normale standard N(0,1) (vectorisée)."""
    return 0.5 * (1.0 + np.vectorize(math.erf)(x / math.sqrt(2.0)))

def distance_kolmogorov(xs):
    """Distance de Kolmogorov entre la loi empirique de xs et N(0,1) :
    d_K = sup_x |F_emp(x) - Phi(x)|, évaluée sur le nuage de points de xs."""
    xs = np.sort(np.asarray(xs, dtype=float))
    n = xs.size
    i = np.arange(1, n + 1)
    d_plus  = np.max(np.abs(i / n - phi(xs)))
    d_minus = np.max(np.abs((i - 1) / n - phi(xs)))
    return float(max(d_plus, d_minus))

def faire_sommes_normalisees(gen, n, m, mu, sigma):
    """m valeurs de (S_n - n*mu)/(sigma*sqrt(n)) où S_n = somme de n tirages i.i.d.
    `gen(n, m)` renvoie un tableau (m, n) de tirages."""
    S = gen(n, m)
    return (S.sum(axis=1) - n * mu) / (sigma * math.sqrt(n))

def echelle_robuste(gen, n, m, mu):
    """Median de |(S_n - n*mu)/sqrt(n)| : une echelle des fluctuations ROBUSTE au
    tirage extremal (insensible a la queue lourde), contrairement a la variance."""
    S = gen(n, m)
    z = (S.sum(axis=1) - n * mu) / math.sqrt(n)
    return float(np.median(np.abs(z)))


## 1. L'énoncé

Soit `X_1, X_2, ...` des variables aléatoires **indépendantes et identiquement distribuées** (i.i.d.), de moyenne `mu = E[X]` et de variance `sigma^2 = Var[X]` **finies**. Posons `S_n = X_1 + ... + X_n`. Le TCL dit que

```
(S_n - n*mu) / (sigma * sqrt(n))   =>   N(0, 1)        (convergence en loi, quand n -> inf)
```

Autrement dit : la **fluctuation** de la somme autour de sa valeur moyenne `n*mu`, rapportée à son échelle naturelle `sigma*sqrt(n)`, a asymptotiquement la forme d'une gaussienne standard.

La promesse est double, et c'est elle que nous allons tester :
- **convergence** : l'écart entre la loi de la somme normalisée et `N(0,1)` tend vers zéro ;
- **universalité** : la forme limite est la même gaussienne, quelle que soit la loi de départ — seul le **taux** dépend de la loi.

Mesurer la convergence, c'est quantifier cet écart. L'outil naturel est la **distance de Kolmogorov** : la plus grande différence entre la fonction de répartition empirique de la somme normalisée et celle de `N(0,1)`. Elle vaut 0 si les deux lois coïncident, et elle est bornée par 1.

In [2]:
# Quatre lois de depart, toutes a VARIANCE FINIE. Pour chacune on note sa moyenne
# mu et son ecart-type sigma (les constantes exactes, pas des approximations).

# Uniforme sur [-1, 1]
def gen_unif(n, m):   return rng.uniform(-1.0, 1.0, (m, n))
mu_unif, sig_unif = 0.0, math.sqrt(1.0 / 3.0)

# Bernoulli(0.3)  (0 ou 1)
def gen_ber(n, m):    return rng.binomial(1, 0.3, (m, n))
p = 0.3
mu_ber, sig_ber = p, math.sqrt(p * (1.0 - p))

# Exponentielle de parametre 1
def gen_exp(n, m):    return rng.exponential(1.0, (m, n))
mu_exp, sig_exp = 1.0, 1.0

# Log-normale exp(N(0,1))  :  E[e^Z] = e^{1/2},  Var = e^2 - e
def gen_logn(n, m):   return rng.lognormal(0.0, 1.0, (m, n))
mu_logn  = math.exp(0.5)
sig_logn = math.sqrt(math.exp(2.0) - math.exp(1.0))

LOIS = [
    ("Uniforme[-1,1]",  gen_unif, mu_unif, sig_unif),
    ("Bernoulli(0.3)",  gen_ber,  mu_ber,  sig_ber),
    ("Exponentielle(1)", gen_exp, mu_exp,  sig_exp),
    ("Log-normale",     gen_logn, mu_logn, sig_logn),
]
for nom, g, mu, sig in LOIS:
    print(f"{nom:16s}  mu = {mu:.4f}   sigma = {sig:.4f}")


Uniforme[-1,1]    mu = 0.0000   sigma = 0.5774
Bernoulli(0.3)    mu = 0.3000   sigma = 0.4583
Exponentielle(1)  mu = 1.0000   sigma = 1.0000
Log-normale       mu = 1.6487   sigma = 2.1612


## 2. Mesurer la convergence : le taux `1/√n`

Pour chaque loi, on tire `m` sommes normalisées pour plusieurs tailles `n` (`n = 10, 40, 160`), et on mesure la distance de Kolmogorov `d_K` à `N(0,1)`.

Si le taux de convergence est `1/√n` — c'est-à-dire `d_K(n) ≃ C / sqrt(n)` — alors le produit `d_K(n)·sqrt(n)` est **indépendant de `n`** : il vaut la constante `C`. C'est la signature que nous cherchons — quatre lois de départ très différentes, un **même taux**. La constante `C`, elle, dépend de la loi.

In [3]:
M = 100000
NS = [10, 40, 160]
entete = "  |  ".join(f"d_K*sqrt(n)  n={n:<3d}" for n in NS)
print("Si la convergence est en 1/sqrt(n), le produit d_K*sqrt(n) est INDEPENDANT de n :")
print()
print(f"{'loi':16s} | {entete}")
print("-" * 64)
for nom, g, mu, sig in LOIS:
    vals = [distance_kolmogorov(faire_sommes_normalisees(g, n, M, mu, sig)) * math.sqrt(n) for n in NS]
    print(f"{nom:16s} | " + "  |  ".join(f"{v:9.4f}" for v in vals))


Si la convergence est en 1/sqrt(n), le produit d_K*sqrt(n) est INDEPENDANT de n :

loi              | d_K*sqrt(n)  n=10   |  d_K*sqrt(n)  n=40   |  d_K*sqrt(n)  n=160
----------------------------------------------------------------
Uniforme[-1,1]   |    0.0146  |     0.0145  |     0.0272


Bernoulli(0.3)   |    0.4769  |     0.4731  |     0.5180
Exponentielle(1) |    0.1320  |     0.1477  |     0.1518


Log-normale      |    0.2938  |     0.3302  |     0.3675


### Lecture du résultat

Le produit `d_K(n)·√n` est, pour chaque loi, **à peu près constant** quand `n` passe de 10 à 160 : c'est exactement l'énoncé `d_K(n) ≃ C / √n`, i.e. le **taux de convergence `1/√n`**. Ce taux est **universel** — il ne dépend ni de la moyenne ni de la variance ni de la forme de la loi de départ.

La **constante** `C` (la ligne, en valeur absolue) dépend, elle, de la loi : c'est le contenu du **théorème de Berry–Esseen**, qui borne `d_K(n) ≤ C'·E|X−mu|³ / (σ³√n)`. Deux lois de même variance mais de **troisième moment** différent convergent à la même vitesse `1/√n` avec des constantes différentes. (Pour l'uniforme, les valeurs s'effondrent vers le plancher de bruit de l'estimation Monte-Carlo de `d_K`, ≈ `1/√m` : sa somme est déjà quasi gaussienne, ce qui masque la constante — un artefact de mesure, pas une dépendance à la loi.)

## 3. Le point central : la variance finie est **porteuse**

Toutes les lois du §2 ont une variance finie. C'est exactement l'hypothèse que le §1 a posée en silence. La question est : **que se passe-t-il si on la retire ?**

Prenons une loi à **queue lourde** dont la variance est **infinie** : la loi de Lomax (Pareto de type II) de forme `alpha = 1.5`. Elle a une moyenne finie (`mu = 2`), mais sa variance n'existe pas.

On ne peut plus normaliser par `sigma*sqrt(n)` (il n'y a pas de `sigma`). On normalise donc par `sqrt(n)` seul, et on regarde l'**échelle des fluctuations** de la somme ainsi normalisée, `z_n = (S_n - n*mu)/sqrt(n)`, quand `n` croît. Pour ne pas se laisser berner par un seul tirage extrême — c'est le propre d'une queue lourde — on mesure une **échelle robuste** : la médiane de `|z_n|`.

- si la loi était dans le domaine d'attraction de la gaussienne, cette échelle se **stabiliserait** (elle tendrait vers une constante, celle de la gaussienne limite) ;
- si la variance est infinie, elle **croît** : l'échelle `sqrt(n)` est trop petite pour capter les fluctuations, qui sont en réalité de l'ordre de `n^{1/alpha}` (plus grand que `sqrt(n) = n^{1/2}` dès que `alpha < 2`). Le rapport des deux, `n^{1/alpha}/n^{1/2} = n^{1/alpha - 1/2}`, est précisément la croissance résiduelle qu'on va mesurer.

C'est la signature mesurable de l'effondrement du TCL.

In [4]:
# Loi de Lomax (Pareto II), forme alpha = 1.5 : moyenne finie, VARIANCE infinie.
ALPHA, XI = 1.5, 1.0
mu_lomax = XI / (ALPHA - 1.0)          # = 2.0
def gen_lomax(n, m):
    # Tirage par quantile (F^{-1}) : x = xi * (1 - u)^{-1/alpha} - xi,  u ~ U(0,1)
    u = rng.random((m, n))
    return XI * ((1.0 - u) ** (-1.0 / ALPHA) - 1.0)

M = 60000
print(f"Echelle robuste median |z_n| = median |(S_n - n*mu)/sqrt(n)|   [m = {M} copies]")
print(f"{'n':>6} | {'uniforme (var finie)':>22} | {'Lomax 1.5 (var infinie)':>26}")
print("-" * 62)
for n in [10, 40, 160, 640]:
    vu = echelle_robuste(gen_unif,  n, M, mu_unif)
    vl = echelle_robuste(gen_lomax, n, M, mu_lomax)
    print(f"{n:>6} | {vu:22.4f} | {vl:26.4f}")


Echelle robuste median |z_n| = median |(S_n - n*mu)/sqrt(n)|   [m = 60000 copies]
     n |   uniforme (var finie) |    Lomax 1.5 (var infinie)
--------------------------------------------------------------
    10 |                 0.3925 |                     2.9382
    40 |                 0.3908 |                     3.9790


   160 |                 0.3906 |                     5.2023


   640 |                 0.3941 |                     6.7492


### Lecture du résultat

La colonne **uniforme** est **plate** (autour de `0.39`, l'échelle de la gaussienne limite `N(0, 1/3)`) : normaliser par `sqrt(n)` donne une fluctuation de **taille constante**, et la somme normalisée converge bien vers une gaussienne — c'est le TCL.

La colonne **Lomax 1.5** **croît** à mesure que `n` grandit : l'échelle des fluctuations de la somme normalisée ne se stabilise pas, elle **grossit** — d'un facteur **2,30 mesuré** entre `n = 10` et `n = 640` (2,94 → 6,75), du même ordre que `64^{1/6} = 2,0`, la croissance résiduelle `n^{1/alpha - 1/2}` prédite pour `alpha = 1.5` (l'écart 2,30 vs 2,0 tient au régime pré-asymptotique). L'échelle `sqrt(n)` est donc la mauvaise : les fluctuations réelles d'une loi à queue lourde de paramètre `alpha` sont de l'ordre de `n^{1/alpha}`, et pour `alpha = 1.5`, `n^{2/3}` croît plus vite que `n^{1/2}`.

**Conclusion** : l'hypothèse de variance finie n'est pas une condition technique qu'on peut ignorer. Elle est **porteuse** — c'est elle qui décide que l'échelle `sqrt(n)` est la bonne et que la limite est gaussienne. Sans elle, le théorème central limite, tel qu'on l'a énoncé, **ne tient plus**. (La somme d'une loi à queue lourde, normalisée par la *bonne* échelle `n^{1/alpha}`, converge vers une **loi stable** non gaussienne — un théorème de convergence généralisé, hors du périmètre du TCL classique.)

## 4. Exercices

In [5]:
# Exercice 1
# TODO etudiant : ecrire la distance de Kolmogorov entre la loi empirique d'un
# echantillon xs et la loi N(0,1).
# # Indice : trier xs ; la supremum est atteint sur les points de l'echantillon ;
# #         comparer i/n (et (i-1)/n) a la fonction de repartition phi aux points triés.
def kolmogorov_etudiant(xs):
    # TODO etudiant
    return None  # TODO etudiant : renvoyer la distance de Kolmogorov (un float)


In [6]:
# Exercice 2
# TODO etudiant : mesurer le taux de convergence. Pour une loi donnee (gen, mu, sigma),
# calculer la distance de Kolmogorov de la somme normalisee pour une liste de valeurs de n,
# et renvoyer la liste des d_K correspondants.
# # Indice : pour chaque n, appeler faire_sommes_normalisees(gen, n, m, mu, sigma) puis
# #         distance_kolmogorov ; collecter dans une liste.
def courbe_convergence_etudiant(gen, mu, sigma, ns, m):
    # TODO etudiant
    return None  # TODO etudiant : renvoyer la liste des distances de Kolmogorov


In [7]:
# Exercice 3
# TODO etudiant : reproduire le contre-exemple. Pour une loi a queue lourde (variance
# infinie), calculer l'echelle robuste median |(S_n - n*mu)/sqrt(n)| pour une liste de n,
# et montrer qu'elle CROIT avec n (au lieu de se stabiliser).
# # Indice : c'est exactement echelle_robuste ; l'appliquer a gen_lomax pour
# #         plusieurs n et verifier que les valeurs montent.
def echelle_robuste_lomax_etudiant(ns, m):
    # TODO etudiant
    return None  # TODO etudiant : renvoyer la liste des echelles robustes (mediane |z_n|)


## En synthèse

Le théorème central limite dit qu'une somme de variables i.i.d. normalisée devient gaussienne. Ce carnet l'a **mesuré** plutôt que subi :

- la convergence est réelle, et son **taux `1/√n` est universel** — indépendant de la loi de départ (seule la constante dépend du troisième moment, Berry–Esseen) ;
- l'hypothèse de **variance finie est porteuse** : un contre-exemple à queue lourde (Lomax 1.5) montre que, sans elle, la somme normalisée ne se stabilise pas en gaussienne — sa variance empirique diverge, et la bonne échelle devient `n^{1/alpha}` (lois stables).

C'est ce qui rend le TCL si utile en pratique : dès qu'on additionne beaucoup de bruits indépendants à variance finie — mesures, erreurs d'échantillonnage, retours sur actifs — la somme « s'oublie » sa forme de départ et devient gaussienne. Et c'est précisément ce qui **n'est pas** garanti dès qu'un bruit à queue lourde (crashes, pics de trafic, gains extrêmes) entre dans la somme.